# 🎨 Rembg AI — Google Colab (A100 GPU)

---

> **📋 Bu notebook hakkında:**  
> Rembg kütüphanesini kullanarak yapay zeka ile arka plan kaldırma işlemi yapar.  
> Tüm ayarlar **form arayüzü** üzerinden yapılır — kod görmene gerek yok.  
> Sonuçlar doğrudan **Google Drive**'a kaydedilir.

---

## 🗺️ Notebook Haritası

| Blok | Ne Yapar? | Çalıştırma |
|------|-----------|------------|
| **🔧 BLOK 1** | Paket kurulumu (versiyon sabitli) | Sadece ilk seferde |
| **💾 BLOK 2** | Google Drive bağlantısı | Her oturumda |
| **🖥️ BLOK 3** | GPU kontrolü & kütüphane yükleme | Her oturumda |
| **⚙️ BLOK 4** | Ayarlar formu (model, kalite, yollar) | Her oturumda |
| **🖼️ BLOK 5** | Tekli resim işleme | İsteğe bağlı |
| **📦 BLOK 6** | Toplu işlem (klasör) | İsteğe bağlı |
| **📊 BLOK 7** | Sonuç raporu | İşlem sonrası |

---

⚠️ **ÖNEMLİ:** Üst menüden `Çalışma Zamanı → GPU türünü değiştir → A100` seçili olduğundan emin ol!

---
## 🔧 BLOK 1 — Paket Kurulumu

> **Ne yapar?**  
> - `rembg` ve tüm bağımlılıklarını **sabit sürümlerle** kurar  
> - `onnxruntime-gpu` kurar → A100 GPU'yu tam performansta kullanır  
> - Sabit sürümler sayesinde gelecekteki güncellemeler bu notebook'u **bozmaz**  
> 
> ⏱️ İlk kurulum ~2-3 dakika sürer. Sonraki oturumlarda tekrar çalıştır.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOK 1: Paket Kurulumu — Versiyon Sabitli                 ║
# ║  Tüm sürümler sabitlenmiştir → güncelleme sorunu yaşanmaz  ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'❌ HATA: {result.stderr[-300:]}')
    return result.returncode == 0

print('📦 Paketler kuruluyor...')

# 1. CUDA uyumlu onnxruntime-gpu (A100 = CUDA 12.x)
print('  ⚡ onnxruntime-gpu...')
run('pip install -q onnxruntime-gpu==1.20.1')

# 2. rembg core (sabit sürüm)
print('  🎨 rembg...')
run('pip install -q "rembg==2.0.59"')

# 3. ipywidgets (form arayüzü)
print('  🎛️ ipywidgets...')
run('pip install -q ipywidgets==8.1.5 tqdm pillow')

# 4. ipywidgets'i Colab için etkinleştir
run('jupyter nbextension enable --py widgetsnbextension')

print()
print('✅ Kurulum tamamlandı!')
print('💡 İpucu: Kernel yeniden başlatmak isteyip istemediğini sorabilir → "Evet" de')

---
## 💾 BLOK 2 — Google Drive Bağlantısı

> **Ne yapar?**  
> - Google Drive'ını Colab'a bağlar  
> - Tüm işlenmiş resimler **Drive'ına otomatik kaydedilir**  
> - `/content/drive/MyDrive/` altına kayıt yapılır  
> - İşlenmiş dosyalar kalıcıdır — oturum kapansa bile kaybolmaz  
> 
> 🔐 Google hesabı ile yetkilendirme isteyecek — izin ver.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOK 2: Google Drive Bağlantısı                           ║
# ╚══════════════════════════════════════════════════════════════╝

from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# Drive bağlantısını doğrula
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Google Drive başarıyla bağlandı!')
    print(f'📁 Drive yolu: /content/drive/MyDrive')
    
    # Drive boyutunu kontrol et
    result = subprocess.run(
        'df -h /content/drive/MyDrive 2>/dev/null | tail -1',
        shell=True, capture_output=True, text=True
    )
    if result.stdout:
        parts = result.stdout.split()
        if len(parts) >= 4:
            print(f'💾 Drive: {parts[1]} toplam, {parts[2]} kullanılıyor, {parts[3]} boş')
else:
    print('❌ Drive bağlanamadı! Lütfen tekrar çalıştır.')

---
## 🖥️ BLOK 3 — GPU Kontrolü & Kütüphaneler

> **Ne yapar?**  
> - Hangi GPU'nun aktif olduğunu kontrol eder  
> - `onnxruntime-gpu`'nun CUDA'yı görüp görmediğini doğrular  
> - A100 görünüyorsa GPU ile işlem yapılır (CPU'dan ~10-20x hızlı!)  
> 
> ⚠️ GPU görünmüyorsa: `Çalışma Zamanı → GPU türünü değiştir → A100`

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOK 3: GPU Kontrolü & Kütüphane İmportları               ║
# ╚══════════════════════════════════════════════════════════════╝

import os, io, json, time, shutil, hashlib, subprocess
from pathlib import Path
from datetime import datetime
from typing import Optional, Tuple

import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output
from PIL import Image
import numpy as np
from tqdm.notebook import tqdm

# ── GPU Bilgisi ──────────────────────────────────────────────────
print('=' * 55)
print('  🖥️  GPU DURUM RAPORU')
print('=' * 55)

gpu_info = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True)
if gpu_info.returncode == 0:
    lines = gpu_info.stdout.split('\n')
    for line in lines:
        if any(x in line for x in ['CUDA', 'A100', 'V100', 'T4', 'Driver', 'MiB', 'GPU']):
            print(f'  {line.strip()}')
else:
    print('  ⚠️  nvidia-smi bulunamadı — GPU yok olabilir')
    print('  ⚠️  Çalışma Zamanı → GPU türünü değiştir → A100')

print()

# ── ONNX Runtime GPU Kontrolü ────────────────────────────────────
try:
    import onnxruntime as ort
    providers = ort.get_available_providers()
    print(f'  🔧 ONNX Runtime: v{ort.__version__}')
    print(f'  📡 Mevcut sağlayıcılar: {providers}')
    if 'CUDAExecutionProvider' in providers:
        print('  ✅ GPU (CUDA) aktif → A100 kullanılıyor!')
        USE_GPU = True
    else:
        print('  ⚠️  GPU bulunamadı → CPU modunda çalışılacak')
        USE_GPU = False
except Exception as e:
    print(f'  ❌ ONNX Runtime hatası: {e}')
    USE_GPU = False

print()

# ── rembg Import ─────────────────────────────────────────────────
try:
    from rembg import remove, new_session
    import rembg
    print(f'  ✅ rembg: v{rembg.__version__}')
except Exception as e:
    print(f'  ❌ rembg import hatası: {e}')
    print('  💡 BLOK 1 hücrelerini tekrar çalıştır!')

print('=' * 55)
print(f'  Mod: {"🚀 GPU (A100)" if USE_GPU else "🐌 CPU (yavaş)"}')
print('=' * 55)

---
## ⚙️ BLOK 4 — Ayarlar Formu

> **Ne yapar?**  
> - Tüm parametreleri görsel formda ayarlarsın  
> - **Model seçimi:** Kullanım amacına göre AI modeli  
> - **Google Drive yolları:** Giriş ve çıkış klasörleri  
> - **Alpha Matting:** Saç, kürk, ince kenarlıklar için iyileştirme  
> - **Skip modu:** Zaten işlenmiş dosyaları atlama  
> 
> 📝 Formu doldurduktan sonra **"💾 Ayarları Kaydet"** butonuna bas!

### 🧠 Model Rehberi

| Model | Tür | Kullanım Alanı | Hız | Kalite |
|-------|-----|----------------|-----|--------|
| `u2net` | 🎯 Genel | Genel amaçlı | 🟡 Orta | ⭐⭐⭐⭐ |
| `u2netp` | ⚡ Hızlı | Toplu/seri işlem | 🟢 Hızlı | ⭐⭐⭐ |
| `u2net_human_seg` | 👤 İnsan | Portre, insan fotoğrafı | 🟡 Orta | ⭐⭐⭐⭐ |
| `u2net_cloth_seg` | 👗 Kıyafet | E-ticaret, tekstil | 🟡 Orta | ⭐⭐⭐⭐ |
| `silueta` | 🖤 Siluet | Sanatsal siluet | 🟢 Hızlı | ⭐⭐⭐ |
| `birefnet-general` | ✨ BiRefNet | **Profesyonel genel** | 🔴 Yavaş | ⭐⭐⭐⭐⭐ |
| `birefnet-portrait` | 🧑 Portre | **Headshot, profil** | 🔴 Yavaş | ⭐⭐⭐⭐⭐ |
| `dis-general-use` | 🌐 DIS | İnce kenarlık, saç, kürk | 🟡 Orta | ⭐⭐⭐⭐⭐ |
| `dis-anime` | 🎌 Anime | Anime, çizgi karakter | 🟡 Orta | ⭐⭐⭐⭐⭐ |
| `bria-rmbg` | 🏢 Ticari | Ürün fotoğrafçılığı | 🟡 Orta | ⭐⭐⭐⭐⭐ |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOK 4: Ayarlar Formu                                     ║
# ║  Tüm parametreleri buradan ayarla, sonra butona bas        ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Stil ────────────────────────────────────────────────────────
display(HTML("""
<style>
.widget-label { font-weight: 600 !important; color: #4a5568 !important; }
.form-section {
  background: linear-gradient(135deg, #667eea10, #764ba210);
  border: 1px solid #667eea40;
  border-radius: 12px;
  padding: 16px 20px;
  margin: 8px 0;
}
.save-btn button {
  background: linear-gradient(135deg, #667eea, #764ba2) !important;
  color: white !important;
  font-weight: 700 !important;
  font-size: 1rem !important;
  border-radius: 8px !important;
  padding: 10px 28px !important;
  border: none !important;
  cursor: pointer !important;
}
</style>
"""))

# ── FORM WİDGET'LARI ─────────────────────────────────────────────

# Başlık
form_title = widgets.HTML('<h2 style="color:#5a67d8;margin:0 0 16px">⚙️ Rembg Ayarlar Paneli</h2>')

# ── Bölüm 1: Model Seçimi ────────────────────────────────────────
sec1 = widgets.HTML('<h3 style="color:#553c9a;margin:4px 0">🧠 Model Seçimi</h3>')

w_model = widgets.Dropdown(
    options=[
        ('🎯 u2net — Genel amaçlı (önerilen başlangıç)',        'u2net'),
        ('⚡ u2netp — En hızlı (seri toplu işlem)',              'u2netp'),
        ('👤 u2net_human_seg — İnsan/portre',                   'u2net_human_seg'),
        ('👗 u2net_cloth_seg — Kıyafet/tekstil',                'u2net_cloth_seg'),
        ('🖤 silueta — Sanatsal siluet',                         'silueta'),
        ('✨ birefnet-general — Yüksek kalite genel',           'birefnet-general'),
        ('✨ birefnet-general-lite — Hızlı yüksek kalite',      'birefnet-general-lite'),
        ('🧑 birefnet-portrait — Headshot / portre (en iyi)',   'birefnet-portrait'),
        ('🔬 birefnet-dis — Detaylı / karmaşık sahneler',       'birefnet-dis'),
        ('📸 birefnet-hrsod — Büyük yüksek çözünürlük',        'birefnet-hrsod'),
        ('🦎 birefnet-cod — Kamuflajlı nesneler',               'birefnet-cod'),
        ('💪 birefnet-massive — Maksimum kalite',               'birefnet-massive'),
        ('🌐 dis-general-use — İnce kenarlık / saç / kürk',    'dis-general-use'),
        ('🎌 dis-anime — Anime / çizgi film karakteri',         'dis-anime'),
        ('🏢 bria-rmbg — Ticari ürün fotoğrafı',               'bria-rmbg'),
    ],
    value='u2net',
    description='AI Modeli:',
    layout=widgets.Layout(width='90%'),
    style={'description_width': '120px'},
)

# ── Bölüm 2: Dosya Yolları ───────────────────────────────────────
sec2 = widgets.HTML('<h3 style="color:#553c9a;margin:12px 0 4px">📁 Google Drive Yolları</h3>')

w_input_path = widgets.Text(
    value='/content/drive/MyDrive/rembg_input',
    description='Giriş Klasörü:',
    placeholder='Resimlerinin bulunduğu Drive klasörü',
    layout=widgets.Layout(width='90%'),
    style={'description_width': '160px'},
)

w_output_path = widgets.Text(
    value='/content/drive/MyDrive/rembg_output',
    description='Çıkış Klasörü:',
    placeholder='İşlenmiş resimlerin kaydedileceği klasör',
    layout=widgets.Layout(width='90%'),
    style={'description_width': '160px'},
)

w_output_suffix = widgets.Text(
    value='_rembg',
    description='Dosya Eki:',
    placeholder='Örn: _rembg → foto_rembg.png',
    layout=widgets.Layout(width='40%'),
    style={'description_width': '160px'},
)

path_info = widgets.HTML('<p style="color:#718096;font-size:0.88rem;margin:4px 0;">💡 Klasörler yoksa otomatik oluşturulur. GB boyutunda dosyalar sorunsuz kaydedilir.</p>')

# ── Bölüm 3: Alpha Matting ───────────────────────────────────────
sec3 = widgets.HTML('<h3 style="color:#553c9a;margin:12px 0 4px">🔬 Alpha Matting (Kenarlık İyileştirme)</h3>')

alpha_info = widgets.HTML('<p style="color:#718096;font-size:0.88rem;margin:4px 0 8px;">💡 Saç, kürk, ince kenarlıklar için aktif et. CPU maliyeti artırır ama kalite çok iyileşir.</p>')

w_alpha = widgets.Checkbox(
    value=False,
    description='Alpha Matting Aktif',
    layout=widgets.Layout(width='90%'),
)

w_fg = widgets.IntSlider(
    value=240, min=0, max=255, step=1,
    description='Ön Plan Eşiği:',
    layout=widgets.Layout(width='80%'),
    style={'description_width': '160px'},
)

w_bg = widgets.IntSlider(
    value=10, min=0, max=255, step=1,
    description='Arka Plan Eşiği:',
    layout=widgets.Layout(width='80%'),
    style={'description_width': '160px'},
)

w_erode = widgets.IntSlider(
    value=10, min=0, max=50, step=1,
    description='Erozyon Boyutu:',
    layout=widgets.Layout(width='80%'),
    style={'description_width': '160px'},
)

# ── Bölüm 4: Ek Seçenekler ──────────────────────────────────────
sec4 = widgets.HTML('<h3 style="color:#553c9a;margin:12px 0 4px">🎨 Ek Seçenekler</h3>')

w_only_mask = widgets.Checkbox(
    value=False,
    description='Sadece maske çıktısı (şeffaf yerine siyah/beyaz)',
    layout=widgets.Layout(width='90%'),
)

w_ppm = widgets.Checkbox(
    value=True,
    description='Maske Post-Process (daha temiz kenarlıklar — önerilen)',
    layout=widgets.Layout(width='90%'),
)

w_skip = widgets.Checkbox(
    value=True,
    description='Zaten işlenmiş dosyaları atla (skip — önerilen)',
    layout=widgets.Layout(width='90%'),
)

skip_info = widgets.HTML('<p style="color:#718096;font-size:0.88rem;margin:2px 0;">💡 Skip aktifse: çıkış klasöründe zaten varsa tekrar işlenmez. Güvenli devam sağlar.</p>')

w_bgcolor = widgets.Checkbox(
    value=False,
    description='Arka plan rengi ekle (şeffaf yerine)',
    layout=widgets.Layout(width='90%'),
)

w_bgcolor_hex = widgets.ColorPicker(
    concise=False,
    description='Arka Plan Rengi:',
    value='#ffffff',
    layout=widgets.Layout(width='50%'),
    style={'description_width': '160px'},
)

w_bgcolor_alpha = widgets.IntSlider(
    value=255, min=0, max=255, step=1,
    description='Renk Saydamlığı:',
    layout=widgets.Layout(width='80%'),
    style={'description_width': '160px'},
)

# ── Bölüm 5: Kayıt Butonu ────────────────────────────────────────
sec5 = widgets.HTML('<h3 style="color:#553c9a;margin:12px 0 4px">💾 Ayarları Uygula</h3>')

save_btn = widgets.Button(
    description='💾 Ayarları Kaydet & Doğrula',
    button_style='',
    layout=widgets.Layout(width='300px', height='44px'),
)
save_btn.add_class('save-btn')

save_out = widgets.Output()

# Global config objesi
CONFIG = {}

def on_save(b):
    with save_out:
        clear_output(wait=True)
        CONFIG.update({
            'model': w_model.value,
            'input_path': w_input_path.value,
            'output_path': w_output_path.value,
            'output_suffix': w_output_suffix.value,
            'alpha': w_alpha.value,
            'fg': w_fg.value,
            'bg': w_bg.value,
            'erode': w_erode.value,
            'only_mask': w_only_mask.value,
            'ppm': w_ppm.value,
            'skip': w_skip.value,
            'use_bgcolor': w_bgcolor.value,
            'bgcolor_hex': w_bgcolor_hex.value,
            'bgcolor_alpha': w_bgcolor_alpha.value,
        })

        # Klasörleri oluştur
        Path(CONFIG['output_path']).mkdir(parents=True, exist_ok=True)

        # Giriş klasörünü kontrol et
        inp = Path(CONFIG['input_path'])
        if not inp.exists():
            inp.mkdir(parents=True, exist_ok=True)
            print(f'📁 Giriş klasörü oluşturuldu: {CONFIG["input_path"]}')
            print('   Resimlerini bu klasöre yükle, sonra işleme bloklarını çalıştır.')
        else:
            exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.gif')
            imgs = [f for f in inp.iterdir() if f.suffix.lower() in exts]
            print(f'📁 Giriş klasörü: {CONFIG["input_path"]}')
            print(f'   → {len(imgs)} resim dosyası bulundu')

        print(f'📁 Çıkış klasörü: {CONFIG["output_path"]}')
        print()
        print('✅ Ayarlar kaydedildi!')
        print(f'   🧠 Model  : {CONFIG["model"]}')
        print(f'   🔬 Alpha  : {"Aktif" if CONFIG["alpha"] else "Kapalı"}')
        print(f'   ⏭️  Skip   : {"Aktif" if CONFIG["skip"] else "Kapalı"}')
        print(f'   🎨 BG Renk: {"Aktif (" + CONFIG["bgcolor_hex"] + ")" if CONFIG["use_bgcolor"] else "Şeffaf"}')
        print(f'   🖥️  GPU    : {"A100 Aktif" if USE_GPU else "CPU"}')

save_btn.on_click(on_save)

# ── Formu Göster ────────────────────────────────────────────────
display(widgets.VBox([
    form_title,
    sec1, w_model,
    widgets.HTML('<hr style="border:1px solid #e2e8f0;margin:8px 0">'),
    sec2, w_input_path, w_output_path, w_output_suffix, path_info,
    widgets.HTML('<hr style="border:1px solid #e2e8f0;margin:8px 0">'),
    sec3, alpha_info, w_alpha, w_fg, w_bg, w_erode,
    widgets.HTML('<hr style="border:1px solid #e2e8f0;margin:8px 0">'),
    sec4, w_only_mask, w_ppm, w_skip, skip_info, w_bgcolor, w_bgcolor_hex, w_bgcolor_alpha,
    widgets.HTML('<hr style="border:1px solid #e2e8f0;margin:8px 0">'),
    sec5, save_btn, save_out,
], layout=widgets.Layout(padding='20px', border='1px solid #e2e8f0', border_radius='16px')))

---
## 🖼️ BLOK 5 — Tekli Resim İşleme

> **Ne yapar?**  
> - Bilgisayarından **tek bir resim** yükler  
> - Seçtiğin model ile arka planı kaldırır  
> - Sonucu hem **ekranda** gösterir hem **Drive'a** kaydeder  
> 
> 💡 İlk kez çalışırken model indirilir (~50MB-1GB arası), biraz beklemeni gerekebilir.  
> 💡 İkinci çalışmada model cache'de → anında başlar.

---

⚠️ **Önce BLOK 4'teki "Ayarları Kaydet" butonuna basmayı unutma!**

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOK 5: Tekli Resim İşleme                                ║
# ║  Yükle → İşle → Gör → Drive'a kayıt                       ║
# ╚══════════════════════════════════════════════════════════════╝

from google.colab import files as colab_files

# ── Yardımcı Fonksiyonlar ────────────────────────────────────────

def hex_to_rgba(hex_color: str, alpha: int = 255):
    """Hex renk kodunu RGBA tuple'a çevirir."""
    h = hex_color.lstrip('#')
    if len(h) == 6:
        return (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16), alpha)
    return None

def get_output_path(input_path: Path, suffix: str, out_dir: str) -> Path:
    """Çıkış dosyası yolunu oluşturur."""
    stem = input_path.stem
    return Path(out_dir) / f'{stem}{suffix}.png'

def process_image(img_bytes: bytes, cfg: dict) -> bytes:
    """Tek bir resmin arka planını kaldırır."""
    session = new_session(cfg['model'])
    bgcolor = None
    if cfg.get('use_bgcolor') and cfg.get('bgcolor_hex'):
        bgcolor = hex_to_rgba(cfg['bgcolor_hex'], cfg.get('bgcolor_alpha', 255))
    return remove(
        img_bytes,
        session=session,
        alpha_matting=cfg['alpha'],
        alpha_matting_foreground_threshold=cfg['fg'],
        alpha_matting_background_threshold=cfg['bg'],
        alpha_matting_erode_size=cfg['erode'],
        only_mask=cfg['only_mask'],
        post_process_mask=cfg['ppm'],
        bgcolor=bgcolor,
    )

def display_comparison(orig_bytes: bytes, result_bytes: bytes, filename: str):
    """Orijinal ve sonucu yan yana gösterir."""
    orig_img = Image.open(io.BytesIO(orig_bytes)).convert('RGBA')
    res_img  = Image.open(io.BytesIO(result_bytes)).convert('RGBA')

    # Küçük boyuta indir (önizleme için)
    max_size = 512
    orig_img.thumbnail((max_size, max_size))
    res_img.thumbnail((max_size, max_size))

    # Yan yana birleştir
    checker = Image.new('RGBA', res_img.size)
    cw, ch = 20, 20
    for y in range(0, ch * 20, ch):
        for x in range(0, cw * 20, cw):
            c = (200, 200, 200, 255) if ((x // cw + y // ch) % 2 == 0) else (255, 255, 255, 255)
            for py in range(ch):
                for px in range(cw):
                    if x + px < res_img.width and y + py < res_img.height:
                        checker.putpixel((x + px, y + py), c)
    bg = Image.new('RGBA', res_img.size)
    bg.paste(checker, (0, 0))
    result_with_bg = Image.alpha_composite(bg, res_img)

    combined_w = orig_img.width + result_with_bg.width + 20
    combined_h = max(orig_img.height, result_with_bg.height) + 40
    combined = Image.new('RGBA', (combined_w, combined_h), (248, 249, 250, 255))
    combined.paste(orig_img.convert('RGBA'), (0, 30))
    combined.paste(result_with_bg, (orig_img.width + 20, 30))

    buf = io.BytesIO()
    combined.convert('RGB').save(buf, format='PNG')
    display(HTML(f'<p style="font-weight:600;color:#553c9a">📸 {filename} — Orijinal (sol) | Sonuç (sağ)</p>'))
    display(IPImage(data=buf.getvalue()))

# ── Upload + İşle Widget'ı ────────────────────────────────────────
upload_btn = widgets.Button(
    description='📤 Resim Seç & Yükle',
    button_style='info',
    layout=widgets.Layout(width='220px', height='44px'),
)
single_out = widgets.Output()

def on_upload_single(b):
    with single_out:
        clear_output(wait=True)

        if not CONFIG:
            print('⚠️ Önce BLOK 4\'teki "Ayarları Kaydet" butonuna bas!')
            return

        print('📤 Dosya seçme penceresi açılıyor...')
        uploaded = colab_files.upload()

        if not uploaded:
            print('❌ Dosya seçilmedi.')
            return

        for filename, content in uploaded.items():
            print(f'\n🔄 İşleniyor: {filename} ({len(content)/1024/1024:.1f} MB)...')
            t0 = time.time()

            try:
                result_bytes = process_image(content, CONFIG)
                elapsed = time.time() - t0

                # Drive'a kaydet
                out_path = get_output_path(
                    Path(filename), CONFIG['output_suffix'], CONFIG['output_path']
                )
                with open(out_path, 'wb') as f:
                    f.write(result_bytes)

                print(f'✅ Tamamlandı! ({elapsed:.1f}s)')
                print(f'💾 Kaydedildi: {out_path}')
                print(f'📦 Dosya boyutu: {len(result_bytes)/1024/1024:.2f} MB')
                print()

                # Önizleme göster
                display_comparison(content, result_bytes, filename)

            except Exception as e:
                import traceback
                print(f'❌ Hata: {e}')
                traceback.print_exc()

upload_btn.on_click(on_upload_single)

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#553c9a">🖼️ Tekli Resim İşleme</h3>'),
    widgets.HTML('<p style="color:#718096">Bilgisayarından resim seç → otomatik işlenir → Drive\'a kaydedilir</p>'),
    upload_btn,
    single_out,
], layout=widgets.Layout(padding='16px', border='1px solid #e2e8f0', border_radius='12px')))

---
## 📦 BLOK 6 — Toplu İşlem (Batch)

> **Ne yapar?**  
> - Drive'daki giriş klasöründeki **tüm resimleri** işler  
> - Zaten işlenmiş dosyaları **skip** eder (atlar) — güvenli devam  
> - Gerçek zamanlı **progress bar** gösterir  
> - İşlem sonunda **özet rapor** verir  
> - GB boyutunda dosyaları bile sorunsuz işler  

> ### 🔄 Skip (Atlama) Mantığı
> Skip aktifse: Çıkış klasöründe `dosya_adi_rembg.png` zaten varsa **tekrar işlenmez**.  
> Bu sayede:
> - İşlem yarıda kesilirse → kaldığı yerden devam eder  
> - Colab oturumu kapanırsa → tekrar çalıştır, sadece eksikler işlenir  
> - Aynı klasöre yeni resim eklenirse → sadece yeniler işlenir  

> ⚠️ **Giriş klasörüne resim eklemek için:**  
> Drive web arayüzünden (`drive.google.com`) yükleyebilir veya aşağıdaki yükleme hücresini kullanabilirsin.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOK 6A: Drive Giriş Klasörüne Toplu Resim Yükleme        ║
# ║  Bilgisayarından → Drive'a → İşlemeye hazır                ║
# ╚══════════════════════════════════════════════════════════════╝

bulk_upload_btn = widgets.Button(
    description='📤 Resimleri Drive\'a Yükle',
    button_style='warning',
    layout=widgets.Layout(width='260px', height='44px'),
)
bulk_upload_out = widgets.Output()

def on_bulk_upload(b):
    with bulk_upload_out:
        clear_output(wait=True)

        if not CONFIG:
            print('⚠️ Önce BLOK 4\'teki "Ayarları Kaydet" butonuna bas!')
            return

        inp_dir = Path(CONFIG['input_path'])
        inp_dir.mkdir(parents=True, exist_ok=True)

        print(f'📁 Yükleme hedefi: {inp_dir}')
        print('📤 Dosya seçme penceresi açılıyor... (birden fazla seçebilirsin)')

        uploaded = colab_files.upload()

        if not uploaded:
            print('❌ Dosya seçilmedi.')
            return

        for filename, content in uploaded.items():
            dest = inp_dir / filename
            with open(dest, 'wb') as f:
                f.write(content)
            print(f'  ✅ {filename} → Drive ({len(content)/1024/1024:.1f} MB)')

        exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.gif')
        total = len([f for f in inp_dir.iterdir() if f.suffix.lower() in exts])
        print(f'\n📊 Giriş klasöründe toplam {total} resim hazır.')
        print('✅ Yükleme tamamlandı! Şimdi toplu işlem bloğunu çalıştırabilirsin.')

bulk_upload_btn.on_click(on_bulk_upload)

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#553c9a">📤 Giriş Klasörüne Toplu Resim Yükleme</h3>'),
    widgets.HTML('<p style="color:#718096">Birden fazla resim seçebilirsin — Drive giriş klasörüne kopyalanır</p>'),
    bulk_upload_btn,
    bulk_upload_out,
], layout=widgets.Layout(padding='16px', border='1px solid #fef3c7', border_radius='12px')))

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOK 6B: Toplu İşlem (Batch)                              ║
# ║  Klasördeki tüm resimler → Skip logic → Drive'a kayıt      ║
# ╚══════════════════════════════════════════════════════════════╝

# İşlem istatistikleri (global)
STATS = {'processed': 0, 'skipped': 0, 'failed': 0, 'total_mb': 0.0, 'start_time': None, 'results': []}

batch_run_btn = widgets.Button(
    description='🚀 Toplu İşlemi Başlat',
    button_style='success',
    layout=widgets.Layout(width='260px', height='48px'),
)

batch_stop_btn = widgets.Button(
    description='⏹️ Durdur',
    button_style='danger',
    layout=widgets.Layout(width='120px', height='48px'),
)

batch_out = widgets.Output()
_stop_flag = {'stop': False}

def on_batch_stop(b):
    _stop_flag['stop'] = True
    with batch_out:
        print('\n⏹️ Durdurma sinyali gönderildi... Mevcut dosya tamamlanınca duracak.')

batch_stop_btn.on_click(on_batch_stop)

def on_batch_run(b):
    with batch_out:
        clear_output(wait=True)
        _stop_flag['stop'] = False

        if not CONFIG:
            print('⚠️ Önce BLOK 4\'teki "Ayarları Kaydet" butonuna bas!')
            return

        inp_dir = Path(CONFIG['input_path'])
        out_dir = Path(CONFIG['output_path'])
        out_dir.mkdir(parents=True, exist_ok=True)

        # Desteklenen uzantılar
        EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif', '.gif'}

        # Giriş klasörünü tara
        if not inp_dir.exists():
            print(f'❌ Giriş klasörü bulunamadı: {inp_dir}')
            print(f'   Drive\'ınızda bu klasörü oluşturun ve resimleri yükleyin.')
            return

        all_files = sorted([f for f in inp_dir.iterdir() if f.suffix.lower() in EXTS])

        if not all_files:
            print(f'❌ Giriş klasöründe resim bulunamadı: {inp_dir}')
            print(f'   Desteklenen formatlar: {', '.join(EXTS)}')
            return

        print('=' * 60)
        print('  📦 TOPLU İŞLEM BAŞLIYOR')
        print('=' * 60)
        print(f'  📁 Giriş : {inp_dir}')
        print(f'  📁 Çıkış : {out_dir}')
        print(f'  🧠 Model : {CONFIG["model"]}')
        print(f'  🔬 Alpha : {"Aktif" if CONFIG["alpha"] else "Kapalı"}')
        print(f'  ⏭️  Skip  : {"Aktif" if CONFIG["skip"] else "Kapalı"}')
        print(f'  🖥️  GPU   : {"A100" if USE_GPU else "CPU"}')
        print(f'  📊 Toplam: {len(all_files)} dosya')
        print('=' * 60)
        print()

        # Model oturumunu önceden yükle (ilk model indirme burada olur)
        print(f'🔄 Model yükleniyor: {CONFIG["model"]}...')
        session = new_session(CONFIG['model'])
        print('✅ Model hazır!\n')

        # Global istatistikleri sıfırla
        STATS.update({'processed': 0, 'skipped': 0, 'failed': 0,
                      'total_mb': 0.0, 'start_time': time.time(), 'results': []})

        bgcolor = None
        if CONFIG.get('use_bgcolor') and CONFIG.get('bgcolor_hex'):
            bgcolor = hex_to_rgba(CONFIG['bgcolor_hex'], CONFIG.get('bgcolor_alpha', 255))

        # Progress bar
        pbar = tqdm(all_files, desc='Resimler işleniyor', unit='resim',
                    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')

        for img_path in pbar:
            if _stop_flag['stop']:
                print('\n⏹️ Kullanıcı tarafından durduruldu.')
                break

            out_path = out_dir / f'{img_path.stem}{CONFIG["output_suffix"]}.png'

            # Skip kontrolü
            if CONFIG['skip'] and out_path.exists():
                pbar.set_postfix({'durum': '⏭️ atlandı'})
                STATS['skipped'] += 1
                STATS['results'].append({'file': img_path.name, 'status': 'skipped'})
                continue

            try:
                file_size_mb = img_path.stat().st_size / 1024 / 1024
                pbar.set_postfix({'dosya': img_path.name[:20], 'boyut': f'{file_size_mb:.1f}MB'})

                t0 = time.time()

                with open(img_path, 'rb') as f:
                    img_bytes = f.read()

                result_bytes = remove(
                    img_bytes,
                    session=session,
                    alpha_matting=CONFIG['alpha'],
                    alpha_matting_foreground_threshold=CONFIG['fg'],
                    alpha_matting_background_threshold=CONFIG['bg'],
                    alpha_matting_erode_size=CONFIG['erode'],
                    only_mask=CONFIG['only_mask'],
                    post_process_mask=CONFIG['ppm'],
                    bgcolor=bgcolor,
                )

                with open(out_path, 'wb') as f:
                    f.write(result_bytes)

                elapsed = time.time() - t0
                out_mb = len(result_bytes) / 1024 / 1024
                STATS['processed'] += 1
                STATS['total_mb'] += out_mb
                STATS['results'].append({
                    'file': img_path.name,
                    'status': 'ok',
                    'elapsed': elapsed,
                    'out_mb': out_mb,
                })
                pbar.set_postfix({'durum': f'✅ {elapsed:.1f}s'})

            except Exception as e:
                STATS['failed'] += 1
                STATS['results'].append({'file': img_path.name, 'status': 'error', 'error': str(e)})
                pbar.set_postfix({'durum': '❌ hata'})
                print(f'\n  ❌ Hata — {img_path.name}: {e}')

        pbar.close()

        # Özet
        total_time = time.time() - STATS['start_time']
        print()
        print('=' * 60)
        print('  📊 İŞLEM TAMAMLANDI')
        print('=' * 60)
        print(f'  ✅ İşlendi  : {STATS["processed"]} dosya')
        print(f'  ⏭️  Atlandı  : {STATS["skipped"]} dosya (zaten vardı)')
        print(f'  ❌ Hata     : {STATS["failed"]} dosya')
        print(f'  📦 Çıkış    : {STATS["total_mb"]:.1f} MB toplam')
        print(f'  ⏱️  Süre     : {total_time:.1f}s ({total_time/60:.1f} dk)')
        if STATS['processed'] > 0:
            print(f'  ⚡ Hız      : {STATS["processed"]/total_time:.1f} resim/sn')
        print(f'  📁 Konum    : {out_dir}')
        print('=' * 60)

        if STATS['failed'] > 0:
            print('\n❌ Hatalı dosyalar:')
            for r in STATS['results']:
                if r['status'] == 'error':
                    print(f'  • {r["file"]}: {r.get("error", "bilinmeyen hata")}')

batch_run_btn.on_click(on_batch_run)

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#553c9a">📦 Toplu İşlem (Batch)</h3>'),
    widgets.HTML('<p style="color:#718096">Giriş klasöründeki tüm resimleri işler. Skip aktifse zaten işlenenler atlanır.</p>'),
    widgets.HBox([batch_run_btn, batch_stop_btn]),
    batch_out,
], layout=widgets.Layout(padding='16px', border='1px solid #c6f6d5', border_radius='12px')))

---
## 📊 BLOK 7 — Sonuç Raporu & Özet

> **Ne yapar?**  
> - Toplu işlem sonrasında detaylı özet gösterir  
> - Başarılı/atlanan/hatalı dosyaları listeler  
> - Çıkış klasöründeki dosya sayısını ve toplam boyutu gösterir  
> - Raporu JSON olarak Drive'a kaydeder (opsiyonel)

---

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOK 7: Sonuç Raporu & Çıkış Klasörü Durumu               ║
# ╚══════════════════════════════════════════════════════════════╝

report_btn = widgets.Button(
    description='📊 Rapor Oluştur',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='44px'),
)

save_report_btn = widgets.Button(
    description='💾 Raporu Drive\'a Kaydet',
    button_style='info',
    layout=widgets.Layout(width='240px', height='44px'),
)

report_out = widgets.Output()

def on_report(b):
    with report_out:
        clear_output(wait=True)

        if not CONFIG:
            print('⚠️ Önce BLOK 4\'teki ayarları kaydet!')
            return

        out_dir = Path(CONFIG['output_path'])

        print('=' * 60)
        print('  📊 ÇIKIŞ KLASÖRÜ RAPORU')
        print('=' * 60)

        if not out_dir.exists():
            print(f'  ❌ Çıkış klasörü henüz oluşturulmadı: {out_dir}')
            return

        png_files = sorted(out_dir.glob('*.png'))
        total_size = sum(f.stat().st_size for f in png_files)

        print(f'  📁 Klasör  : {out_dir}')
        print(f'  📊 Dosya   : {len(png_files)} PNG')
        print(f'  💾 Toplam  : {total_size/1024/1024:.1f} MB ({total_size/1024/1024/1024:.2f} GB)')
        print()

        # Son işlemden istatistik (varsa)
        if STATS.get('results'):
            print('  📋 Son Toplu İşlem Sonuçları:')
            print(f'     ✅ İşlendi  : {STATS["processed"]}')
            print(f'     ⏭️  Atlandı  : {STATS["skipped"]}')
            print(f'     ❌ Hata     : {STATS["failed"]}')
            if STATS.get('start_time'):
                elapsed = time.time() - STATS['start_time']
                print(f'     ⏱️  Süre     : {elapsed:.0f}s')
            print()

        # Son 10 dosyayı listele
        if png_files:
            print('  📋 Son 10 İşlenmiş Dosya:')
            for f in png_files[-10:]:
                size_kb = f.stat().st_size / 1024
                mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
                print(f'     • {f.name:<40} {size_kb:>8.0f} KB  {mtime}')

        print('=' * 60)

def on_save_report(b):
    with report_out:
        if not CONFIG or not STATS.get('results'):
            print('\n⚠️ Önce toplu işlemi çalıştır, sonra raporu kaydet!')
            return

        report = {
            'timestamp': datetime.now().isoformat(),
            'config': CONFIG,
            'stats': {
                'processed': STATS['processed'],
                'skipped': STATS['skipped'],
                'failed': STATS['failed'],
                'total_mb': round(STATS['total_mb'], 2),
                'gpu': USE_GPU,
            },
            'results': STATS['results'],
        }

        report_path = Path(CONFIG['output_path']) / f'rembg_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(report, f, ensure_ascii=False, indent=2)

        print(f'\n✅ Rapor kaydedildi: {report_path}')

report_btn.on_click(on_report)
save_report_btn.on_click(on_save_report)

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#553c9a">📊 Sonuç Raporu</h3>'),
    widgets.HBox([report_btn, save_report_btn]),
    report_out,
], layout=widgets.Layout(padding='16px', border='1px solid #e2e8f0', border_radius='12px')))

---
## 🛠️ BLOK 8 — Bakım & Yardım

> **Ne yapar?**  
> - Model cache temizleme  
> - Colab disk kullanım kontrolü  
> - Sorun giderme araçları  

### ❓ Sık Karşılaşılan Sorunlar

| Sorun | Çözüm |
|-------|-------|
| `No module named 'rembg'` | BLOK 1'i tekrar çalıştır |
| GPU görünmüyor | Çalışma zamanı → GPU türünü değiştir → A100 |
| Drive bağlanmıyor | BLOK 2'yi tekrar çalıştır |
| Model indirilmiyor | İnternet bağlantısını kontrol et |
| Colab oturumu kapandı | BLOK 2-3-4'ü çalıştır, sonra BLOK 6B'yi çalıştır (skip devam eder) |
| RAM yetersiz | Daha küçük batch'lerle çalış veya u2netp modelini seç |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOK 8: Bakım Araçları                                    ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Disk Kullanımı ───────────────────────────────────────────────
print('💾 COLAB DISK KULLANIMI:')
import subprocess
r = subprocess.run('df -h / /content 2>/dev/null', shell=True, capture_output=True, text=True)
print(r.stdout)

# ── Model Cache Konumu ───────────────────────────────────────────
cache_dir = Path.home() / '.u2net'
if cache_dir.exists():
    models = list(cache_dir.glob('*.onnx'))
    total_cache = sum(m.stat().st_size for m in models) / 1024 / 1024 / 1024
    print(f'\n🧠 MODEL CACHE: {cache_dir}')
    for m in models:
        size_mb = m.stat().st_size / 1024 / 1024
        print(f'   • {m.name:<40} {size_mb:>6.0f} MB')
    print(f'   Toplam: {total_cache:.2f} GB')
    print()
    print('💡 Not: Oturum kapandığında model cache silinir → tekrar indirilir.')
    print('   Bunu önlemek için model cache klasörünü Drive\'a kopyalayabilirsin:')
    print('   !cp -r ~/.u2net /content/drive/MyDrive/.u2net_cache')
    print('   Sonraki oturumda: !cp -r /content/drive/MyDrive/.u2net_cache ~/.u2net')
else:
    print('\n🧠 Henüz model indirilmemiş (ilk işlemde indirilecek).')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BLOK 8B: Model Cache → Drive'a Kaydet / Drive'dan Yükle   ║
# ║  Model indirme süresini sıfıra indirir!                    ║
# ╚══════════════════════════════════════════════════════════════╝

cache_save_btn = widgets.Button(
    description='💾 Cache → Drive\'a Kaydet',
    button_style='info',
    layout=widgets.Layout(width='260px', height='44px'),
)

cache_load_btn = widgets.Button(
    description='⬇️ Drive\'dan Cache Yükle',
    button_style='warning',
    layout=widgets.Layout(width='260px', height='44px'),
)

cache_out = widgets.Output()

def on_cache_save(b):
    with cache_out:
        clear_output(wait=True)
        cache_dir = Path.home() / '.u2net'
        if not cache_dir.exists() or not list(cache_dir.glob('*.onnx')):
            print('❌ Kaydedilecek model cache bulunamadı.')
            print('   Önce en az bir işlem yap — model otomatik indirilecek.')
            return
        drive_cache = Path('/content/drive/MyDrive/.u2net_model_cache')
        print(f'💾 Cache kaydediliyor... {cache_dir} → {drive_cache}')
        shutil.copytree(str(cache_dir), str(drive_cache), dirs_exist_ok=True)
        models = list(drive_cache.glob('*.onnx'))
        total_gb = sum(m.stat().st_size for m in models) / 1024**3
        print(f'✅ {len(models)} model Drive\'a kaydedildi ({total_gb:.2f} GB)')
        print(f'   Yol: {drive_cache}')

def on_cache_load(b):
    with cache_out:
        clear_output(wait=True)
        drive_cache = Path('/content/drive/MyDrive/.u2net_model_cache')
        if not drive_cache.exists():
            print('❌ Drive\'da model cache bulunamadı.')
            print('   Önce "Cache → Drive\'a Kaydet" butonunu kullan.')
            return
        cache_dir = Path.home() / '.u2net'
        cache_dir.mkdir(exist_ok=True)
        print(f'⬇️ Cache yükleniyor... {drive_cache} → {cache_dir}')
        shutil.copytree(str(drive_cache), str(cache_dir), dirs_exist_ok=True)
        models = list(cache_dir.glob('*.onnx'))
        print(f'✅ {len(models)} model yüklendi → model indirme adımı atlanacak!')
        for m in models:
            print(f'   • {m.name}')

cache_save_btn.on_click(on_cache_save)
cache_load_btn.on_click(on_cache_load)

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#553c9a">🧠 Model Cache Yönetimi</h3>'),
    widgets.HTML('<p style="color:#718096">Modeli Drive\'a kaydet → oturumlar arası hızlı başlat (indirme yok!)</p>'),
    widgets.HBox([cache_save_btn, cache_load_btn]),
    cache_out,
], layout=widgets.Layout(padding='16px', border='1px solid #bee3f8', border_radius='12px')))